In [1]:
import numpy as np
import pickle
from qiskit_nature.second_q.operators import SpinOp
from qiskit_nature.second_q.mappers import LogarithmicMapper
n_qubit = 1
dim     = 2**n_qubit
mapper = LogarithmicMapper()
t = 0.5
nld = 11
lds = np.linspace(0,1,num=nld)
Hld = []
i = 0
for ld in lds:
    h = SpinOp({
        "X_0": t,  
        "Z_0": -1.0 + 2.0* ld
    },
    spin=1/2
    )
    Hld.append(2*mapper.map(h.simplify())) # 2 is due to spin 1/2
    print(Hld[i])
    i +=1
h = SpinOp({
    "Z_0": 2.0
},
spin=1/2
)
Hp = 2*mapper.map(h.simplify()) # 2 is due to spin 1/2

# exact eigenvalues
ell  = np.zeros((nld,dim),dtype=float)
vll  = np.zeros((nld,dim,dim),dtype=complex)
for ild in range(nld):
    El, Vl = np.linalg.eigh(Hld[ild].to_matrix())
    indx = np.argsort(El.real)
    for i in range(dim):
        ell[ild,i]   = El[indx[i]].real
        vll[ild,:,i] = Vl[:,indx[i]]

X = np.zeros((2,2),dtype=complex)
Y = np.zeros((2,2),dtype=complex)
Z = np.zeros((2,2),dtype=complex)

X[0,1] = 1; X[1,0] = 1
Y[0,1] = -1j; Y[1,0] = 1j
Z[0,0] = 1; Z[1,1] = -1

# exact results
norm_exact   = np.ones((nld,dim),dtype=float)
E_exact      = np.zeros((nld,dim),dtype=float)
E_exact[0,:] = ell[0,:]
for k in range(dim):
    phi = vll[0,:,k]
    for ild in range(1,nld):
        Proj_matrix = np.outer(vll[ild,:,k],vll[ild,:,k].conj())
        phi = Proj_matrix@phi
        norm_exact[ild,k] = phi.conj()@phi
        E_exact[ild,k] = phi.conj()@Hld[ild].to_matrix()@phi/norm_exact[ild,k]
X_exact = np.zeros((nld,dim),dtype=float)
Y_exact = np.zeros((nld,dim),dtype=float)
Z_exact = np.zeros((nld,dim),dtype=float)
for ild in range(nld):
    for k in range(dim):
        X_exact[ild,k] = vll[ild,:,k].conj().transpose()@X@vll[ild,:,k]
        Y_exact[ild,k] = vll[ild,:,k].conj().transpose()@Y@vll[ild,:,k]
        Z_exact[ild,k] = vll[ild,:,k].conj().transpose()@Z@vll[ild,:,k]

0.5 * X
- 1.0 * Z
0.5 * X
- 0.8 * Z
0.5 * X
- 0.6 * Z
0.5 * X
- 0.3999999999999999 * Z
0.5 * X
- 0.19999999999999996 * Z
0.5 * X
0.5 * X
+ 0.20000000000000018 * Z
0.5 * X
+ 0.40000000000000013 * Z
0.5 * X
+ 0.6000000000000001 * Z
0.5 * X
+ 0.8 * Z
0.5 * X
+ 1.0 * Z


/home/mchan/Doc/venv_qiskit/lib/python3.10/site-packages/qiskit_nature/second_q/mappers/qubit_mapper.py:188: PauliSumOpDeprecationWarning: PauliSumOp is deprecated as of version 0.6.0 and support for them will be removed no sooner than 3 months after the release. Instead, use SparsePauliOp. You can switch to SparsePauliOp immediately, by setting `qiskit_nature.settings.use_pauli_sum_op` to `False`.
  qubit_ops[name] = self._map_single(second_q_op, register_length=register_length)
/tmp/ipykernel_24643/986796965.py:57: ComplexWarning: Casting complex values to real discards the imaginary part
  norm_exact[ild,k] = phi.conj()@phi
/tmp/ipykernel_24643/986796965.py:58: ComplexWarning: Casting complex values to real discards the imaginary part
  E_exact[ild,k] = phi.conj()@Hld[ild].to_matrix()@phi/norm_exact[ild,k]
/tmp/ipykernel_24643/986796965.py:57: ComplexWarning: Casting complex values to real discards the imaginary part
  norm_exact[ild,k] = phi.conj()@phi
/tmp/ipykernel_24643/98679696

In [2]:
with open('norm.E.exact','w') as file_:
    s = '# λ , norm_k^2, std(norm_k^2), E_k, std(E_k), k= 1, .., dim'
    s += '\n'
    file_.write(s)
    for ild in range(nld):
        s = '{:.16e}'.format(lds[ild])
        for k in range(dim):
            s += '  {:.16e}  {:.16e}  {:.16e}  {:.16e}'.format(norm_exact[ild,k],0.0,E_exact[ild,k],0.0)
        s += '\n'
        file_.write(s)


In [3]:
with open('xyz.exact','w') as file_:
    s = '# λ , X, std(X), Y, std(Y), Z, std(Z), F, std(F)'
    for ild in range(nld):
        s = '{:.16e}'.format(lds[ild])
        for k in range(1):
            s += '  {:.16e}  {:.16e}  {:.16e}  {:.16e}  {:.16e}  {:.16e}  {:.16e}  {:.16e}'.format(X_exact[ild,k],0.0,Y_exact[ild,k],0.0,Z_exact[ild,k],0.0,1.0,0.0)
        s += '\n'
        file_.write(s)
